In [2]:
# %%
# =============================================================================
# NOTEBOOK: 01_eda_D_TDC.ipynb
# Paper D (TDC: Text-Disclosure Credit) — EDA before any modeling
#
# Verify the dataset supports the core hypothesis BEFORE assuming it:
#   (1) sample sizes, bankruptcy rate (imbalance), train/test split
#   (2) text availability & length (MD&A, Risk Factors) — missing/empty?
#   (3) numeric-only baseline signal (so we can later isolate text's added value)
#
# Data: text-based bankruptcy dataset, Mendeley DOI 10.17632/stf3kg7fw3
#   10-K MD&A + Risk Factors + numeric financials + Twitter; Bankruptcy label.
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
TEXT_DIR = ROOT / "data" / "raw" / "text_bankruptcy"
TAB = ROOT / "artifacts" / "tables"
TAB.mkdir(parents=True, exist_ok=True)


def rel(p):
    try: return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError: return Path(p).name


# --- Load 10-K train/test (text + numeric) + labels --------------------------
Xtr = pd.read_csv(TEXT_DIR / "NUM10K_X_train_2021June.csv")
Xte = pd.read_csv(TEXT_DIR / "NUM10K_X_test_2021June.csv")
ytr = pd.read_csv(TEXT_DIR / "NUM10K_y_train_2021June.csv")
yte = pd.read_csv(TEXT_DIR / "NUM10K_y_test_2021June.csv")
Xtr["Bankruptcy"] = ytr["Bankruptcy"].values
Xte["Bankruptcy"] = yte["Bankruptcy"].values

print("── (1) Size & imbalance ──")
for name, d in [("train", Xtr), ("test", Xte)]:
    n, pos = len(d), int(d.Bankruptcy.sum())
    print(f"  {name}: {n:,} firms, {pos} bankrupt ({pos/n:.2%})")

# --- (2) Text availability & length ------------------------------------------
print("\n── (2) Text availability (MD&A, RiskFactors) ──")
for col in ["MDandA", "RiskFactors"]:
    for name, d in [("train", Xtr), ("test", Xte)]:
        s = d[col].fillna("").astype(str)
        empty = (s.str.len() < 50).mean()          # effectively empty
        wc = s.str.split().str.len()
        print(f"  {col} [{name}]: empty/near-empty {empty:.1%}, "
              f"median words {int(wc.median())}, "
              f"p90 words {int(wc.quantile(0.9))}")

# --- (3) Numeric columns present (for numeric baseline) ----------------------
num_cols = [c for c in Xtr.columns
            if c not in ("cik", "RiskFactors", "MDandA", "date", "Bankruptcy")]
print(f"\n── (3) Numeric features: {len(num_cols)} columns ──")
print("  ", num_cols[:20])
print(f"  numeric missingness (train, mean): "
      f"{Xtr[num_cols].isna().mean().mean():.1%}")

# --- (4) Bankruptcy rate vs text length (early signal check) -----------------
print("\n── (4) Do bankrupt firms have different text length? (early hint) ──")
Xtr["_mdna_wc"] = Xtr["MDandA"].fillna("").astype(str).str.split().str.len()
Xtr["_rf_wc"] = Xtr["RiskFactors"].fillna("").astype(str).str.split().str.len()
print(Xtr.groupby("Bankruptcy")[["_mdna_wc", "_rf_wc"]].median().round(0).to_string())
print("  (large gaps would hint text carries distress signal; not conclusive)")

# Save a compact EDA summary
summary = pd.DataFrame({
    "split": ["train", "test"],
    "n": [len(Xtr), len(Xte)],
    "bankrupt": [int(Xtr.Bankruptcy.sum()), int(Xte.Bankruptcy.sum())],
    "bankruptcy_rate": [Xtr.Bankruptcy.mean(), Xte.Bankruptcy.mean()],
})
summary.round(4).to_csv(TAB / "D_eda_summary.csv", index=False)
print(f"\n  → saved: {rel(TAB / 'D_eda_summary.csv')}")

── (1) Size & imbalance ──
  train: 111 firms, 55 bankrupt (49.55%)
  test: 111 firms, 56 bankrupt (50.45%)

── (2) Text availability (MD&A, RiskFactors) ──
  MDandA [train]: empty/near-empty 0.0%, median words 9781, p90 words 23639
  MDandA [test]: empty/near-empty 0.0%, median words 11230, p90 words 30436
  RiskFactors [train]: empty/near-empty 0.0%, median words 8768, p90 words 20792
  RiskFactors [test]: empty/near-empty 0.0%, median words 8572, p90 words 20829

── (3) Numeric features: 39 columns ──
   ['sale', 'ch', 'invt', 'ap', 'act', 'ebit', 'gp', 'oancf', 'at', 're', 'ni', 'ebitda', 'dd1', 'dltt', 'lct', 'Working capital/Total assets', 'CurrentRatio', 'Current Assets/Total Assets ', '(Current Assets-Inventory)/Tot. Assets', 'Current Liabilities/Total Assets']
  numeric missingness (train, mean): 0.0%

── (4) Do bankrupt firms have different text length? (early hint) ──
            _mdna_wc  _rf_wc
Bankruptcy                  
0             7913.0  7926.0
1            12368.0 